In [ ]:
import numpy as np
from nn import *

In [ ]:

data = np.loadtxt("emnist-balanced-train.csv", delimiter=",", dtype=int)

In [4]:
y_train = data[:, 0]
x_train = data[:, 1:].astype(np.float32) / 255.0


In [5]:
print(y_train.shape)

(112800,)


In [6]:
# making startified validation set

y_vals = np.unique(y_train)
validation_indices = []
rng1 = np.random.default_rng()

for val in y_vals:
    indices = np.where(y_train == val)[0]
    rng1.shuffle(indices)
    validation_indices.extend(indices[: int(len(indices)*0.1)])

validation_indices = np.array(validation_indices, dtype=int)
rng1.shuffle(validation_indices)

x_validation = x_train[validation_indices]
y_validation = y_train[validation_indices]

x_train = np.delete(x_train, validation_indices, axis=0)
y_train = np.delete(y_train, validation_indices)







In [7]:
x_train = cp.asarray(x_train)
y_train = cp.asarray(y_train)

x_validation = cp.asarray(x_validation)
y_validation =cp.asarray(y_validation)

In [8]:
perm = cp.random.permutation(len(y_train))

x_train = x_train[perm]
y_train = y_train[perm]

In [ ]:

dense1 = Layer_Dense(784, 512, weight_reg_l2=1e-4)
activation1 = Activation_ReLU()
dropout1 = Layer_Dropout(0.2)

dense2 = Layer_Dense(512, 256, weight_reg_l2=1e-4)
activation2 = Activation_ReLU()
dropout2 = Layer_Dropout(0.2)

dense3 = Layer_Dense(256, 47)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()
optimizer = Optimizer_Adam(decay = 1e-5, learning_rate=0.001)


total_epochs = 40
for epoch in range(1, total_epochs+1):

    perm = cp.random.permutation(len(y_train))

    x_train = x_train[perm]
    y_train = y_train[perm]

    batch_size = 128
    i = 0
    x = x_train[i:i+batch_size]
    y = y_train[i:i+batch_size]
    total_samples = len(x_train)

    training = True
    iteration = 1


    correct_preds = 0
    total_spb = 0      # total samples per batch used to calcualte acc, loss over a full epoch
    total_loss = 0.0     # accumulated loss (without reg) over a full epoch
    total_reg_loss = 0.0   # accumulated loss with reg over a full epoch
    while(True):

        dense1.forward(x)
        activation1.forward(dense1.output)
        dropout1.forward(activation1.output)

        dense2.forward(dropout1.output)
        activation2.forward(dense2.output)
        dropout2.forward(activation2.output)

        dense3.forward(dropout2.output)

        loss = loss_activation.forward(dense3.output, y)

        reg = (loss_activation.loss.regularization_loss(dense1) +
        loss_activation.loss.regularization_loss(dense2) +
        loss_activation.loss.regularization_loss(dense3))

        reg_loss = loss + reg
        predictions = cp.argmax(loss_activation.output, axis=1)

        if(len(y.shape) == 2):
            y = cp.argmax(y, axis=1)
        accuracy = cp.mean(predictions==y)


        correct_preds += cp.sum(predictions == y)
        total_spb += len(y)
        total_loss += loss * len(y)
        total_reg_loss += reg_loss * len(y)


        loss_activation.backward(loss_activation.output, y)
        dense3.backward(loss_activation.dinputs)

        dropout2.backward(dense3.dinputs)
        activation2.backward(dropout2.dinputs)
        dense2.backward(activation2.dinputs)

        dropout1.backward(dense2.dinputs)
        activation1.backward(dropout1.dinputs)
        dense1.backward(activation1.dinputs)

        optimizer.pre_update_params()
        optimizer.update_params(dense1)
        optimizer.update_params(dense2)
        optimizer.update_params(dense3)
        optimizer.post_update_params()

        i += batch_size

        if(not training or (i==total_samples)):
            break

        elif(i+batch_size > total_samples):

            x = x_train[i: total_samples]
            y = y_train[i: total_samples]
            training = False

        else:

            x = x_train[i:i+batch_size]
            y = y_train[i:i+batch_size]

        iteration += 1


    epoch_acc = (correct_preds / total_spb).item()
    epoch_loss = (total_loss / total_spb).item()
    epoch_reg_loss = (total_reg_loss / total_spb).item()


    print("====================================================================")
    print("TRAINING SUMARRY PER EPOCH")
    print(f"EPOCH: {epoch} | " +
          f"ACC: {epoch_acc} | " +
          f"ORIGINAL LOSS: {epoch_loss} | " +
          f"WITH REG LOSS: {epoch_reg_loss}")
    print("====================================================================")

# VALIDATING AFTER EVERY EPOCH

    x = x_validation
    y = y_validation

    dense1.forward(x)
    activation1.forward(dense1.output)

    dense2.forward(activation1.output)
    activation2.forward(dense2.output)

    dense3.forward(activation2.output)

    loss = loss_activation.forward(dense3.output, y)
    predictions = cp.argmax(loss_activation.output, axis=1)

    if(len(y.shape) == 2):
        y = cp.argmax(y, axis=1)
    accuracy = cp.mean(predictions==y).item()
    loss = loss.item()

    print("====================================================================")
    print("Validation SUMARRY PER EPOCH")
    print(f"EPOCH: {epoch} | " +
          f"ACC: {accuracy} | " +
          f"LOSS: {loss}")
    print("====================================================================")
    print("\n")





TRAINING SUMARRY PER EPOCH
EPOCH: 1 | ACC: 0.6646473601260835 | ORIGINAL LOSS: 1.135925054550171 | WITH REG LOSS: 1.2742741107940674
Validation SUMARRY PER EPOCH
EPOCH: 1 | ACC: 0.7874113475177305 | LOSS: 0.6640862822532654


TRAINING SUMARRY PER EPOCH
EPOCH: 2 | ACC: 0.7837470449172577 | ORIGINAL LOSS: 0.6696789264678955 | WITH REG LOSS: 0.8152621388435364
Validation SUMARRY PER EPOCH
EPOCH: 2 | ACC: 0.8174645390070922 | LOSS: 0.55374675989151


TRAINING SUMARRY PER EPOCH
EPOCH: 3 | ACC: 0.8113376674546887 | ORIGINAL LOSS: 0.5701777935028076 | WITH REG LOSS: 0.7185203433036804
Validation SUMARRY PER EPOCH
EPOCH: 3 | ACC: 0.8289007092198581 | LOSS: 0.5116421580314636


TRAINING SUMARRY PER EPOCH
EPOCH: 4 | ACC: 0.8248029944838455 | ORIGINAL LOSS: 0.5185303688049316 | WITH REG LOSS: 0.6681383848190308
Validation SUMARRY PER EPOCH
EPOCH: 4 | ACC: 0.8351063829787234 | LOSS: 0.4929579794406891


TRAINING SUMARRY PER EPOCH
EPOCH: 5 | ACC: 0.8328605200945627 | ORIGINAL LOSS: 0.48908981680870

In [ ]:


test_data = np.loadtxt("emnist-balanced-test.csv", delimiter=",", dtype=int)
y_test = test_data[:, 0]
x_test = test_data[:, 1:].astype(np.float32) / 255.0

x_test = cp.asarray(x_test)
y_test = cp.asarray(y_test)

# Forward pass, no dropout, no reg
dense1.forward(x_test)
activation1.forward(dense1.output)

dense2.forward(activation1.output)
activation2.forward(dense2.output)

dense3.forward(activation2.output)

test_loss = loss_activation.forward(dense3.output, y_test)
predictions = cp.argmax(loss_activation.output, axis=1)

if y_test.ndim == 1:
    y_true = y_test
else:
    y_true = cp.argmax(y_test, axis=1)

test_accuracy = cp.mean(predictions == y_true).item()
test_loss = test_loss.item()

print("====================================================================")
print("TEST SET RESULTS (No reg | No dropout)")
print(f"ACC: {test_accuracy}  LOSS: {test_loss}")
print("====================================================================")

TEST SET RESULTS (No reg | No dropout)
ACC: 0.8595744680851064  LOSS: 0.41424351930618286
